In [1]:
language = "english"

dump_file = f"/mnt/Velocity_Vault/Wiki_Dump/Dump/{language}_dump.xml"
raw_corpus_file = f"/mnt/Velocity_Vault/Wiki_Dump/Corpus/{language}_corpus.txt"
corpus_file = f"/mnt/Velocity_Vault/Wiki_Dump/Corpus/{language}_processed.txt"

co_occur_matrix_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_matrix.npy"

pearson_matrix_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_pearson_matrix.npy"

svd_matrix_U_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_svd_matrix_U.npy"
svd_matrix_S_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_svd_matrix_S.npy"
svd_matrix_V_file = f"/mnt/Velocity_Vault/Wiki_Dump/Memory/{language}_svd_matrix_V.npy"

svd_save_files=[svd_matrix_U_file,svd_matrix_S_file,svd_matrix_V_file]

In [2]:
from collections import Counter
from tqdm import tqdm

def get_vocabulary(filename, min_freq):
    """Reads corpus file, counts word occurrences.
    Returns a list of words sorted by frequency (highest first)."""
    with open(filename, 'r') as file:
        words = file.read().strip().split()
        
    freq_dict = Counter(words)
    
    word_freq_pairs = [(word, freq) for word, freq in freq_dict.items() if freq >= min_freq]
    
    sorted_word_freq_pairs = sorted(word_freq_pairs, key=lambda x: x[1], reverse=True)
    
    vocab_list = [word for word, _ in sorted_word_freq_pairs]
    
    return vocab_list

vocabulary = get_vocabulary(corpus_file, 250)
print("Vocabulary Size - ", len(vocabulary))

Vocabulary Size -  22590


In [3]:
import numpy as np

def compute_svd(pcm_file,save_files):
    """Compute truncated SVD of co-occurrence or correlation matrix"""
    P = np.load(pcm_file).astype(np.float32)
    
    U, sigma, Vt = np.linalg.svd(P, full_matrices=False)
    
    np.save(save_files[0],np.array(U))
    np.save(save_files[1],np.array(sigma))
    np.save(save_files[2],np.array(Vt))

    return U, sigma, Vt

def compute_embeddings(save_files, embedding_dim):
    
    U = np.load(save_files[0]).astype(np.float32)
    sigma = np.load(save_files[1]).astype(np.float32)
    Vt = np.load(save_files[2]).astype(np.float32)
    
    U_k = U[:, :embedding_dim]
    
    # V_k = Vt[:embedding_dim, :].T
    Sigma_k = np.diag(sigma[:embedding_dim])

    sqrt_sigma = np.diag(np.sqrt(sigma[:embedding_dim]))

    return U_k @ Sigma_k   



In [4]:
import numpy as np
from tabulate import tabulate

def display_svd_similarities(vocab_list, target_words, word_vectors):
    """Display similarities using SVD for both finding and scoring words"""
    
    results = []
    target_indices = [vocab_list.index(word) for word in target_words]
    
    for target_idx in target_indices:
        target_vector = word_vectors[target_idx]
        
        # Calculate cosine similarities in SVD space
        dot_products = word_vectors @ target_vector
        norms = np.linalg.norm(word_vectors, axis=1)
        target_norm = np.linalg.norm(target_vector)
        svd_similarities = dot_products / (norms * target_norm)
        
        # Get top 5 similar words (excluding itself)
        top_indices = np.argsort(svd_similarities)[::-1][1:6]
        similar_words = [vocab_list[idx] for idx in top_indices]
        similarity_scores = [svd_similarities[idx] for idx in top_indices]
        
        # Format the results
        similar_lines = [f"{word} ({score:.4f})" for word, score in zip(similar_words, similarity_scores)]
        results.append("\n".join(similar_lines))
    
    # EXACT same table format
    table_data = [results]
    print(tabulate(table_data, headers=target_words, tablefmt="grid"))


In [5]:
# U, sigma, Vt = compute_svd(pearson_matrix_file, svd_save_files)

word_vectors = compute_embeddings(svd_save_files, embedding_dim=300)

In [6]:
# Display results with exact same format
target_words = vocabulary[4250:4255]
target_words = ['brother', 'daughter', 'mother', 'china', 'minister']
display_svd_similarities(vocabulary, target_words, word_vectors)

+------------------+-----------------------+------------------+----------------------+----------------------+
| brother          | daughter              | mother           | china                | minister             |
+==================+=======================+==================+======================+======================+
| ahmed (1.0000)   | niece (1.0000)        | epstein (1.0000) | georgia (1.0000)     | justice (1.0000)     |
| mathew (1.0000)  | alumnus (1.0000)      | dino (1.0000)    | yemen (1.0000)       | consort (1.0000)     |
| raymond (1.0000) | acquaintance (1.0000) | father (1.0000)  | brentwood (1.0000)   | scientology (1.0000) |
| antoni (1.0000)  | breath (1.0000)       | argento (1.0000) | haifa (1.0000)       | images (1.0000)      |
| isaac (1.0000)   | sq (1.0000)           | wen (1.0000)     | sanctuaries (1.0000) | visions (1.0000)     |
+------------------+-----------------------+------------------+----------------------+----------------------+
